# Dependances

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler #Le volume domine beaucoup trop
#from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import LSTM, Dropout, Dense, BatchNormalization
#from tensorflow.keras.callbacks import EarlyStopping

#import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

['C:\\Users\\utilisateur\\AppData\\Local\\Programs\\Python\\Python312', 'C:\\Users\\utilisateur\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages']


# Dataset

- 400+ cryptos, 1 minute resolution : https://www.kaggle.com/datasets/tencars/392-crypto-currency-pairs-at-minute-resolution?resource=download

In [ ]:
df = pd.read_csv('datams.csv')
df.head()

Suppression des crypto's avec moins d'"Hours" heures de données

In [ ]:
hours = 48

block_counts = df.groupby('crypto')['block'].count()
print(sum(block_counts < hours)) #1 mois c'est 43 200

#cryptos_to_drop = block_counts[block_counts < hours].index
#df = df[~df["crypto"].isin(cryptos_to_drop)]

#print(f"Nombre de cryptos supprimées : {len(cryptos_to_drop)}")
#print(f"Taille du DataFrame filtré : {df.shape}")

# Entrainement Transformer

## Embedding

Nous devons creer des embedding afin d'apprendre au modele que chaque crypto peut agir d'une maniere différente

In [ ]:
crypto_mapping = {crypto: idx for idx, crypto in enumerate(df['crypto'].unique())}

df['crypto_id'] = df['crypto'].map(crypto_mapping)
df

In [ ]:
print("CUDA disponible :", torch.cuda.is_available())

# Vérifier quel GPU est utilisé
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
else:
    print("Aucun GPU disponible")

# LSTM

In [ ]:
w_size = 48

'''Fonction pour créer des séquences temporelles de plusieurs heure'''

def create_sequences(data, target, w_size):

    sequences, targets = [], []
    for i in range(len(data) - w_size):
        sequences.append(data[i:i+w_size]) #X
        targets.append(target[i+w_size]) #Y
    return np.array(sequences), np.array(targets)

#
X = df[['open', 'close', 'high', 'low']].values
y = df['close'].values  # Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler_x, scaler_y = MinMaxScaler(), MinMaxScaler()

X_train, y_train  = scaler_x.fit_transform(X_train), scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
X_test, y_test = scaler_x.transform(X_test), scaler_y.transform(y_test.reshape(-1, 1)).flatten()

X_train_seq, y_train_seq = create_sequences(X_train, y_train, w_size)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, w_size)

print(f"X_train_seq shape: {X_train_seq.shape}, X_test_seq shape: {X_test_seq.shape}")
print(f"y_train_seq shape: {y_train_seq.shape}, y_test_seq shape: {y_test_seq.shape}")

In [ ]:
print("GPU est disponible :", tf.config.list_physical_devices('GPU'))

In [ ]:
input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])

model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(256, return_sequences=True),
        Dense(128, activation='tanh'),
        LSTM(64, return_sequences=False),
        Dense(32, activation='tanh'),
        Dense(1)  # Sortie
])

model.compile(optimizer='adam', loss='huber_loss', metrics=['mae']) #ou RMSprop
model.summary()

In [ ]:
history = model.fit(
    X_train_seq, y_train_seq,
    epochs=2,
    batch_size=64,
    validation_data=(X_test_seq, y_test_seq),
    verbose=1,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)